In [ ]:
# !pip install torch_geometric

In [ ]:
# !pip install torch-scatter -f https://data.pyg.org/whl/torch-2.5.1+cu121.html

In [ ]:
# import all libraries needed downstream
import os
import gc
import wandb
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from copy import deepcopy
import scipy
from sklearn.metrics import mean_squared_error
import math
import networkx as nx
import seaborn as sns
import time
import itertools
from torch.nn import Linear
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import degree
from torch_geometric.nn import ChebConv, GraphConv, GCNConv, TAGConv, GATConv
from torch_geometric.data import Data
from torch.utils.data import TensorDataset
from torch_geometric.loader import DataLoader
import torch_scatter
from typing import Dict, Tuple, List, Optional
import matplotlib.pyplot as plt
from torch_scatter import scatter_softmax, scatter_sum

In [ ]:
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
!nvidia-smi

In [ ]:
# system_size = 14

In [ ]:
# Load the data
def load_system(system_size):
    
    data = np.load(f'/home/oarowolo/workfile/OPFData/{system_size}bus_nminusone_combined_dataset.npz',allow_pickle=True)
    
    # Get all keys
    print("Available keys in the dataset:")
    for key in data.files:
        # Print the key and its array shape
        print(f"{key}: shape {data[key].shape}")
    return data

In [ ]:
def compute_gandb(edge_inputs):

    line_r = edge_inputs[:,4:5]
    line_x = edge_inputs[:,5:6]

    line_g = line_r/(line_r**2 + line_x**2)
    line_b = -line_x/(line_r**2 + line_x**2)

    return line_g, line_b


In [ ]:
def get_B_matrix(N, edges, edge_weights):
    # Create a zero tensor of shape (N,N)
    B_matrix = np.zeros((N, N))
    
    # Unpack the edges into source and destination nodes
    sources, destinations = zip(*edges)
    
    # Use advanced indexing to place weights in the right spots
    B_matrix[sources, destinations] = edge_weights.squeeze()
    B_matrix[destinations, sources] = edge_weights.squeeze()
    return B_matrix

In [ ]:
def adjacency_to_laplacian(B_adj):

    # Ensure matrix is square
    assert B_adj.shape[0] == B_adj.shape[1], "Input must be square"

    # Copy to avoid modifying original
    B_laplacian = B_adj.copy()

    # Set diagonal as row sum of adjacency (i.e., degree)
    np.fill_diagonal(B_laplacian, -B_adj.sum(axis=1))

    return B_laplacian


In [ ]:
# Solution 2: More efficient - precompute full pseudoinverse diagonal
import scipy.sparse as sp
import scipy.sparse.linalg as spla
def compute_row_stats_from_laplacian(L, tol=1e-8):
    """
    More efficient approach: precompute pseudoinverse diagonal, then process rows.
    """
    L = sp.csc_matrix(L)
    N = L.shape[0]
    
    # Remove reference node and compute reduced inverse
    keep = np.arange(N - 1)
    L_reduced = L[np.ix_(keep, keep)]
    L_reduced_inv = spla.inv(L_reduced.tocsc()).toarray()
    
    # Expand to full pseudoinverse
    L_plus = np.zeros((N, N))
    L_plus[np.ix_(keep, keep)] = L_reduced_inv
    
    # Apply projection
    I = np.eye(N)
    ones = np.ones((N, N)) / N
    L_plus = (I - ones) @ L_plus @ (I - ones)
    
    # Compute effective resistance matrix and find global max
    diag = np.diag(L_plus)
    R = diag[:, None] + diag[None, :] - 2 * L_plus
    
    # Find global maximum (excluding diagonal which should be ~0)
    mask = ~np.eye(N, dtype=bool)
    global_max = np.max(R[mask])
    
    # Normalize and compute statistics
    R_normalized = R / global_max
    stats_matrix = np.zeros((N, 5))
    
    for i in range(N):
        row_no_diag = R_normalized[i, mask[i]]
        stats_matrix[i, 0] = np.mean(row_no_diag)
        stats_matrix[i, 1] = np.median(row_no_diag)
        stats_matrix[i, 2] = np.std(row_no_diag)
        stats_matrix[i, 3] = np.max(row_no_diag)
        stats_matrix[i, 4] = np.min(row_no_diag)
    
    return stats_matrix

In [ ]:
def load_grid_data(data):      
        
    # Access grid input features
    grid_bus = data['grid_bus']  
    grid_generator = data['grid_generator']
    grid_load = data['grid_load']
    grid_shunt = data['grid_shunt']
    grid_ac_line_features = data['grid_ac_line_features']
    grid_transformer_features = data['grid_transformer_features']
    grid_ac_line_receivers = data['grid_ac_line_receivers']
    grid_ac_line_senders = data['grid_ac_line_senders']
    grid_transformer_senders = data['grid_transformer_senders']
    grid_transformer_receivers = data['grid_transformer_receivers']
    grid_generator_link_senders = data['grid_generator_link_senders']
    grid_generator_link_receivers = data['grid_generator_link_receivers']
    solution_bus = data['solution_bus']  
    solution_generator = data['solution_generator'] 
    solution_objective = data['metadata_objective']
    solution_objective = solution_objective.reshape(-1,1)
    grid_load_link_receivers = data['grid_load_link_receivers']
    grid_load_link_senders = data['grid_load_link_senders']
    grid_shunt_link_receivers = data['grid_shunt_link_receivers']
    grid_shunt_link_senders = data['grid_shunt_link_senders']
    
    return grid_bus, grid_generator, grid_load, grid_shunt, grid_ac_line_features,grid_ac_line_senders,grid_ac_line_receivers, grid_transformer_features, grid_transformer_senders, grid_transformer_receivers, solution_bus, solution_generator, solution_objective, grid_load_link_receivers, grid_load_link_senders, grid_shunt_link_receivers, grid_shunt_link_senders, grid_generator_link_senders, grid_generator_link_receivers 

In [ ]:
def process_grid_data(grid_bus, grid_generator, grid_load, grid_shunt, grid_ac_line_features, grid_ac_line_senders, grid_ac_line_receivers, grid_transformer_features, grid_transformer_senders, grid_transformer_receivers, solution_bus, solution_generator, solution_objective, grid_load_link_receivers, grid_load_link_senders, grid_shunt_link_receivers, grid_shunt_link_senders, grid_generator_link_senders, grid_generator_link_receivers, system_size):

    grid_bus_list = []
    grid_generator_list = []
    grid_load_list = []
    grid_shunt_list = []
    grid_ac_line_features_list  = []
    grid_transformer_features_list = []
    solution_bus_list = []
    solution_generator_list = []
    grid_ac_line_senders_list = []
    grid_ac_line_receivers_list = []
    grid_transformer_senders_list = []
    grid_transformer_receivers_list = []
    load_indices_list = []
    shunt_indices_list = []
    generator_indices_list = []
    master_branch_list = []
    pe_list = []
    
    for k in tqdm(range(grid_bus.shape[0]), desc="Data Processing Progress"): 
        
        generator_indices = grid_generator_link_receivers[k].astype(int)
        generator_indices_list.append(generator_indices)
        load_indices = grid_load_link_receivers[k].astype(int)
        load_indices_list.append(load_indices)
        shunt_indices = grid_shunt_link_receivers[k].astype(int)
        shunt_indices_list.append(shunt_indices)
        
        grid_bus_k = grid_bus[k].astype(np.float32)
        grid_bus_list.append(grid_bus_k)
        grid_generator_k = grid_generator[k].astype(np.float32)
        grid_generator_list.append(grid_generator_k)
        grid_load_k = grid_load[k].astype(np.float32)
        grid_load_list.append(grid_load_k)
        grid_shunt_k = grid_shunt[k].astype(np.float32)
        grid_shunt_list.append(grid_shunt_k)
        
        solution_bus_k = solution_bus[k].astype(np.float32)
        solution_bus_list.append(solution_bus_k)
        solution_generator_k = solution_generator[k].astype(np.float32)
        solution_generator_list.append(solution_generator_k)
    
        grid_transformer_sender_k = grid_transformer_senders[k].astype(np.float32)
        grid_transformer_senders_list.append(grid_transformer_sender_k)
        grid_transformer_receiver_k = grid_transformer_receivers[k].astype(np.float32)
        grid_transformer_receivers_list.append(grid_transformer_receiver_k)
        grid_ac_line_sender_k = grid_ac_line_senders[k].astype(np.float32)
        grid_ac_line_senders_list.append(grid_ac_line_sender_k)
        grid_ac_line_receiver_k = grid_ac_line_receivers[k].astype(np.float32)
        grid_ac_line_receivers_list.append(grid_ac_line_receiver_k)
    
        grid_transformer_features_k = grid_transformer_features[k].astype(np.float32)
        grid_transformer_features_list.append(grid_transformer_features_k)
        grid_ac_line_features_k = grid_ac_line_features[k].astype(np.float32)
        grid_ac_line_features_list.append(grid_ac_line_features_k)
        
        branch_list = list(zip(grid_ac_line_senders[k], grid_ac_line_receivers[k]))
        transformer_list = list(zip(grid_transformer_senders[k], grid_transformer_receivers[k]))
        for j in transformer_list:
            branch_list.append(j)
        master_branch_list.append(branch_list)

        # pe_list = torch.load(f'bus_{system_size}_normed_pe_encodings.pt')
    
        ################################ this is where we create the positional encoding stuff
    
        edge_inputs = np.zeros((len(branch_list),11))
        
        edge_inputs[:grid_ac_line_features_k.shape[0],:9] = grid_ac_line_features_k  # rearranging edge inputs to align for transformers and transmission lines
        edge_inputs[grid_ac_line_features_k.shape[0]:,:2] =  grid_transformer_features_k[:,:2]
        edge_inputs[grid_ac_line_features_k.shape[0]:,2:4] =  grid_transformer_features_k[:,9:]
        edge_inputs[grid_ac_line_features_k.shape[0]:,4:9] =  grid_transformer_features_k[:,2:7]
        edge_inputs[grid_ac_line_features_k.shape[0]:,9:] =  grid_transformer_features_k[:,7:9]
        edge_inputs[:grid_ac_line_features_k.shape[0],9:10] = 1.0
    
        edge_g, edge_b = compute_gandb(edge_inputs)
        B_weighted = get_B_matrix(system_size, branch_list,edge_b)
        b_mat = B_weighted
        B_lap = adjacency_to_laplacian(b_mat)
        raw_PE = compute_row_stats_from_laplacian(B_lap)
        bus_pe = torch.tensor(raw_PE,dtype=torch.float)
        pe_list.append([bus_pe])

    return  grid_bus_list, grid_generator_list, grid_load_list, grid_shunt_list, grid_ac_line_features_list, grid_transformer_features_list, solution_bus_list, solution_generator_list, grid_ac_line_senders_list, grid_ac_line_receivers_list, grid_transformer_senders_list, grid_transformer_receivers_list, load_indices_list, shunt_indices_list, generator_indices_list, master_branch_list, pe_list
                

In [ ]:
batch_size = 64

In [ ]:
from torch_geometric.data import HeteroData

def create_grid_hetero_data(
    grid_bus,                    
    grid_generator,              
    grid_load,                   
    grid_shunt,                  
    grid_ac_line_features,       
    grid_transformer_features,   
    grid_ac_line_senders,        
    grid_ac_line_receivers,      
    grid_transformer_senders,    
    grid_transformer_receivers,  
    generator_indices,          
    load_indices,                
    shunt_indices,               
    solution_bus,                
    solution_generator,
    pe,
    batch_idx=0
):
    
    # Create a new HeteroData instance
    data = HeteroData()
    
    # Process a single batch if specified, otherwise we'd need to handle batching differently
    if batch_idx is not None:
        # Extract features for the specified batch
        bus_features = torch.tensor(grid_bus[batch_idx], dtype=torch.float)
        generator_features = torch.tensor(grid_generator[batch_idx], dtype=torch.float)
        load_features = torch.tensor(grid_load[batch_idx], dtype=torch.float)
        shunt_features = torch.tensor(grid_shunt[batch_idx], dtype=torch.float)


        bus_pe = pe[batch_idx][0].to(torch.float)
        
        ac_line_features = torch.tensor(grid_ac_line_features[batch_idx], dtype=torch.float)
        transformer_features = torch.tensor(grid_transformer_features[batch_idx], dtype=torch.float)
        
        # Extract solution values for the specified batch
        bus_solutions = torch.tensor(solution_bus[batch_idx], dtype=torch.float)
        generator_solutions = torch.tensor(solution_generator[batch_idx], dtype=torch.float)
        
        # Add node features
        data['bus'].x = bus_features
        data['generator'].x = generator_features
        data['load'].x = load_features
        data['shunt'].x = shunt_features

        #create global IDs for nodes to make indexing of transformer easier
        data['bus'].graph_id = torch.full((bus_features.shape[0],), batch_idx, dtype=torch.long)
        data['generator'].graph_id = torch.full((generator_features.shape[0],), batch_idx, dtype=torch.long)
        data['load'].graph_id = torch.full((load_features.shape[0],), batch_idx, dtype=torch.long)
        data['shunt'].graph_id = torch.full((shunt_features.shape[0],), batch_idx, dtype=torch.long)

        ####use indices specially
        g_indices = generator_indices[batch_idx]
        s_indices = shunt_indices[batch_idx]
        l_indices = load_indices[batch_idx]

        #Add node positional encoding
        data['bus'].pe = bus_pe
        data['generator'].pe = bus_pe[g_indices]  ## added arbitrary values to distinguish buses from special nodes
        data['load'].pe = bus_pe[l_indices]
        data['shunt'].pe = bus_pe[s_indices]
        
        # Add solution values as target values (y)
        data['bus'].y = bus_solutions
        data['generator'].y = generator_solutions
        
        # Add edge indices and features for AC lines (bus to bus)
        senders = torch.tensor(grid_ac_line_senders[batch_idx].flatten(), dtype=torch.long)
        receivers = torch.tensor(grid_ac_line_receivers[batch_idx].flatten(), dtype=torch.long)
        edge_index = torch.stack([senders, receivers], dim=0)
        data['bus', 'ac_line', 'bus'].edge_index = edge_index
        data['bus', 'ac_line', 'bus'].edge_attr = ac_line_features
        
        # Add edge indices and features for transformers (bus to bus)
        senders = torch.tensor(grid_transformer_senders[batch_idx].flatten(), dtype=torch.long)
        receivers = torch.tensor(grid_transformer_receivers[batch_idx].flatten(), dtype=torch.long)
        edge_index = torch.stack([senders, receivers], dim=0)
        data['bus', 'transformer', 'bus'].edge_index = edge_index
        data['bus', 'transformer', 'bus'].edge_attr = transformer_features
        
        # Add pseudo-edges from generators to buses
        gen_to_bus = torch.tensor(generator_indices[batch_idx].flatten(), dtype=torch.long)
        gen_indices = torch.arange(len(gen_to_bus), dtype=torch.long)
        gen_edge_index = torch.stack([gen_indices, gen_to_bus], dim=0)
        data['generator', 'connects_to', 'bus'].edge_index = gen_edge_index
        data['generator', 'connects_to', 'bus'].edge_attr = torch.ones((len(gen_to_bus), 3))
        
        # Add pseudo-edges from loads to buses
        load_to_bus = torch.tensor(load_indices[batch_idx].flatten(), dtype=torch.long)
        load_index = torch.arange(len(load_to_bus), dtype=torch.long)
        load_edge_index = torch.stack([load_index, load_to_bus], dim=0)
        data['load', 'connects_to', 'bus'].edge_index = load_edge_index
        data['load', 'connects_to', 'bus'].edge_attr = torch.ones((len(load_to_bus), 3))
        
        # Add pseudo-edges from shunts to buses
        shunt_to_bus = torch.tensor(shunt_indices[batch_idx].flatten(), dtype=torch.long)
        shunt_index = torch.arange(len(shunt_to_bus), dtype=torch.long)
        shunt_edge_index = torch.stack([shunt_index, shunt_to_bus], dim=0)
        data['shunt', 'connects_to', 'bus'].edge_index = shunt_edge_index
        data['shunt', 'connects_to', 'bus'].edge_attr = torch.ones((len(shunt_to_bus), 3))
    
    else:
        # Handle all batches (would require batching approach)
        raise NotImplementedError("Processing all batches at once is not implemented in this example")
    
    return data


def create_dataloader(
    grid_bus,
    grid_generator,
    grid_load,
    grid_shunt,
    grid_ac_line_features,
    grid_transformer_features,
    grid_ac_line_senders,
    grid_ac_line_receivers,
    grid_transformer_senders,
    grid_transformer_receivers,
    generator_indices,
    load_indices,
    shunt_indices,
    solution_bus,
    solution_generator,
    pe,
    batch_size=batch_size,
    shuffle = True
):
    
    # Create a list of HeteroData objects
    dataset = []
    
    for i in range(len(grid_bus)):  # Process up to 1000 samples for this example
        data = create_grid_hetero_data(
            grid_bus, 
            grid_generator,
            grid_load,
            grid_shunt,
            grid_ac_line_features,
            grid_transformer_features,
            grid_ac_line_senders,
            grid_ac_line_receivers,
            grid_transformer_senders,
            grid_transformer_receivers,
            generator_indices,
            load_indices,
            shunt_indices,
            solution_bus,
            solution_generator,
            pe,
            batch_idx=i
        )
        dataset.append(data)
    
    # Create a DataLoader
    # loader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,num_workers=min(8, torch.get_num_threads()))
    
    return dataset

In [ ]:
def calculate_angle_differences(angles, edges):
  
    # Ensure angles are a NumPy array
    angles = np.asarray(angles)
    
    # Initialize array to store angle differences
    angle_differences = np.zeros(len(edges), dtype=np.float32)
    
    # Calculate angle differences for each edge
    for i, (node1, node2) in enumerate(edges):
        angle_differences[i] = angles[node2] - angles[node1]
    
    return angle_differences

In [ ]:
def train_val_test_split(train_ratio=0.9, val_ratio=0.05, test_ratio=0.05, seed=42):

    # Set random seed for reproducibility
    torch.manual_seed(seed)

    # Shuffle indices
    num_samples = 300000
    indices = torch.randperm(num_samples)

    # Compute split sizes
    train_size = int(train_ratio * num_samples)
    val_size = int(val_ratio * num_samples)

    # reduce the actual training size to just 20k samples per system
    
    reduced_train_size = 20000
    # Split indices
    train_indices = indices[:reduced_train_size]
    val_indices = indices[train_size:train_size + val_size]
    test_indices = indices[train_size + val_size:]

    return train_indices, val_indices, test_indices

In [ ]:
train_indices, val_indices, test_indices = train_val_test_split()

In [ ]:
def make_training_datasets(grid_bus_list, grid_generator_list, grid_load_list, grid_shunt_list, grid_ac_line_features_list, grid_transformer_features_list, solution_bus_list, solution_generator_list, grid_ac_line_senders_list, grid_ac_line_receivers_list, grid_transformer_senders_list, grid_transformer_receivers_list, load_indices_list, shunt_indices_list, generator_indices_list, master_branch_list, pe_list):

    train_grid_bus =  list(grid_bus_list[i] for i in train_indices)
    train_grid_generator=  list(grid_generator_list[i] for i in train_indices)
    train_grid_load=  list(grid_load_list[i] for i in train_indices)
    train_grid_shunt=  list(grid_shunt_list[i] for i in train_indices)
    train_grid_ac_line_features=  list(grid_ac_line_features_list[i] for i in train_indices)
    train_grid_transformer_features=  list(grid_transformer_features_list[i] for i in train_indices)
    train_grid_ac_line_senders=  list(grid_ac_line_senders_list[i] for i in train_indices)
    train_grid_ac_line_receivers=  list(grid_ac_line_receivers_list[i] for i in train_indices)
    train_grid_transformer_senders=  list(grid_transformer_senders_list[i] for i in train_indices)
    train_grid_transformer_receivers=  list(grid_transformer_receivers_list[i] for i in train_indices)
    train_generator_indices=  list(generator_indices_list[i] for i in train_indices)
    train_load_indices=  list(load_indices_list[i] for i in train_indices)
    train_shunt_indices=  list(shunt_indices_list[i] for i in train_indices)
    train_solution_bus=  list(solution_bus_list[i] for i in train_indices)
    train_solution_generator=  list(solution_generator_list[i] for i in train_indices)
    train_bus_pe = [pe_list[i] for i in train_indices]

    train_dataset= create_dataloader(train_grid_bus,
                                train_grid_generator,
                                train_grid_load,
                                train_grid_shunt,
                                train_grid_ac_line_features,
                                train_grid_transformer_features,
                                train_grid_ac_line_senders,
                                train_grid_ac_line_receivers,
                                train_grid_transformer_senders,
                                train_grid_transformer_receivers,
                                train_generator_indices,
                                train_load_indices,
                                train_shunt_indices,
                                train_solution_bus,
                                train_solution_generator,
                                train_bus_pe,
                                batch_size=batch_size,
                                shuffle = True)

    return train_dataset 

In [ ]:
def make_validation_datasets(grid_bus_list, grid_generator_list, grid_load_list, grid_shunt_list, grid_ac_line_features_list, grid_transformer_features_list, solution_bus_list, solution_generator_list, grid_ac_line_senders_list, grid_ac_line_receivers_list, grid_transformer_senders_list, grid_transformer_receivers_list, load_indices_list, shunt_indices_list, generator_indices_list, master_branch_list, pe_list):

    validate_grid_bus =  list(grid_bus_list[i] for i in val_indices)
    validate_grid_generator=  list(grid_generator_list[i] for i in val_indices)
    validate_grid_load=  list(grid_load_list[i] for i in val_indices)
    validate_grid_shunt=  list(grid_shunt_list[i] for i in val_indices)
    validate_grid_ac_line_features=  list(grid_ac_line_features_list[i] for i in val_indices)
    validate_grid_transformer_features=  list(grid_transformer_features_list[i] for i in val_indices)
    validate_grid_ac_line_senders=  list(grid_ac_line_senders_list[i] for i in val_indices)
    validate_grid_ac_line_receivers=  list(grid_ac_line_receivers_list[i] for i in val_indices)
    validate_grid_transformer_senders=  list(grid_transformer_senders_list[i] for i in val_indices)
    validate_grid_transformer_receivers=  list(grid_transformer_receivers_list[i] for i in val_indices)
    validate_generator_indices=  list(generator_indices_list[i] for i in val_indices)
    validate_load_indices=  list(load_indices_list[i] for i in val_indices)
    validate_shunt_indices=  list(shunt_indices_list[i] for i in val_indices)
    validate_solution_bus=  list(solution_bus_list[i] for i in val_indices)
    validate_solution_generator=  list(solution_generator_list[i] for i in val_indices)
    validate_bus_pe = list(pe_list[i] for i in val_indices)

    val_dataset =  create_dataloader(validate_grid_bus,
                                    validate_grid_generator,
                                    validate_grid_load,
                                    validate_grid_shunt,
                                    validate_grid_ac_line_features,
                                    validate_grid_transformer_features,
                                    validate_grid_ac_line_senders,
                                    validate_grid_ac_line_receivers,
                                    validate_grid_transformer_senders,
                                    validate_grid_transformer_receivers,
                                    validate_generator_indices,
                                    validate_load_indices,
                                    validate_shunt_indices,
                                    validate_solution_bus,
                                    validate_solution_generator,
                                    validate_bus_pe,
                                    batch_size=batch_size,
                                    shuffle = False)

    return val_dataset 

In [ ]:
data_14 = load_system(14)
data_30 = load_system(30)
data_57 = load_system(57)
data_118 = load_system(118)
data_500 = load_system(500)

In [ ]:
 grid_bus_14, grid_generator_14, grid_load_14, grid_shunt_14, grid_ac_line_features_14, grid_ac_line_senders_14, grid_ac_line_receivers_14, grid_transformer_features_14, grid_transformer_senders_14, grid_transformer_receivers_14, solution_bus_14, solution_generator_14, solution_objective_14, grid_load_link_receivers_14, grid_load_link_senders_14, grid_shunt_link_receivers_14, grid_shunt_link_senders_14, grid_generator_link_senders_14, grid_generator_link_receivers_14  = load_grid_data(data_14)

In [ ]:
 grid_bus_30, grid_generator_30, grid_load_30, grid_shunt_30, grid_ac_line_features_30, grid_ac_line_senders_30, grid_ac_line_receivers_30, grid_transformer_features_30, grid_transformer_senders_30, grid_transformer_receivers_30, solution_bus_30, solution_generator_30, solution_objective_30, grid_load_link_receivers_30, grid_load_link_senders_30, grid_shunt_link_receivers_30, grid_shunt_link_senders_30, grid_generator_link_senders_30, grid_generator_link_receivers_30 = load_grid_data(data_30)

In [ ]:
 grid_bus_57, grid_generator_57, grid_load_57, grid_shunt_57, grid_ac_line_features_57, grid_ac_line_senders_57, grid_ac_line_receivers_57, grid_transformer_features_57, grid_transformer_senders_57, grid_transformer_receivers_57, solution_bus_57, solution_generator_57, solution_objective_57, grid_load_link_receivers_57, grid_load_link_senders_57, grid_shunt_link_receivers_57, grid_shunt_link_senders_57, grid_generator_link_senders_57, grid_generator_link_receivers_57 = load_grid_data(data_57)

In [ ]:
 grid_bus_118, grid_generator_118, grid_load_118, grid_shunt_118, grid_ac_line_features_118, grid_ac_line_senders_118, grid_ac_line_receivers_118, grid_transformer_features_118, grid_transformer_senders_118, grid_transformer_receivers_118, solution_bus_118, solution_generator_118, solution_objective_118, grid_load_link_receivers_118, grid_load_link_senders_118, grid_shunt_link_receivers_118, grid_shunt_link_senders_118, grid_generator_link_senders_118, grid_generator_link_receivers_118 = load_grid_data(data_118)

In [ ]:
 grid_bus_500, grid_generator_500, grid_load_500, grid_shunt_500, grid_ac_line_features_500, grid_ac_line_senders_500, grid_ac_line_receivers_500, grid_transformer_features_500, grid_transformer_senders_500, grid_transformer_receivers_500, solution_bus_500, solution_generator_500, solution_objective_500, grid_load_link_receivers_500, grid_load_link_senders_500, grid_shunt_link_receivers_500, grid_shunt_link_senders_500, grid_generator_link_senders_500, grid_generator_link_receivers_500 = load_grid_data(data_500)

In [ ]:
grid_bus_list_14, grid_generator_list_14, grid_load_list_14, grid_shunt_list_14, grid_ac_line_features_list_14, grid_transformer_features_list_14, solution_bus_list_14, solution_generator_list_14, grid_ac_line_senders_list_14, grid_ac_line_receivers_list_14, grid_transformer_senders_list_14, grid_transformer_receivers_list_14, load_indices_list_14, shunt_indices_list_14, generator_indices_list_14, master_branch_list_14, pe_list_14 = process_grid_data(grid_bus_14, grid_generator_14, grid_load_14, grid_shunt_14, grid_ac_line_features_14, grid_ac_line_senders_14, grid_ac_line_receivers_14, grid_transformer_features_14, grid_transformer_senders_14, grid_transformer_receivers_14, solution_bus_14, solution_generator_14, solution_objective_14, grid_load_link_receivers_14, grid_load_link_senders_14, grid_shunt_link_receivers_14, grid_shunt_link_senders_14, grid_generator_link_senders_14, grid_generator_link_receivers_14, 14) 

In [ ]:
grid_bus_list_30, grid_generator_list_30, grid_load_list_30, grid_shunt_list_30, grid_ac_line_features_list_30, grid_transformer_features_list_30, solution_bus_list_30, solution_generator_list_30, grid_ac_line_senders_list_30, grid_ac_line_receivers_list_30, grid_transformer_senders_list_30, grid_transformer_receivers_list_30, load_indices_list_30, shunt_indices_list_30, generator_indices_list_30, master_branch_list_30, pe_list_30 = process_grid_data(grid_bus_30, grid_generator_30, grid_load_30, grid_shunt_30, grid_ac_line_features_30, grid_ac_line_senders_30, grid_ac_line_receivers_30, grid_transformer_features_30, grid_transformer_senders_30, grid_transformer_receivers_30, solution_bus_30, solution_generator_30, solution_objective_30, grid_load_link_receivers_30, grid_load_link_senders_30, grid_shunt_link_receivers_30, grid_shunt_link_senders_30, grid_generator_link_senders_30, grid_generator_link_receivers_30, 30) 

In [ ]:
grid_bus_list_57, grid_generator_list_57, grid_load_list_57, grid_shunt_list_57, grid_ac_line_features_list_57, grid_transformer_features_list_57, solution_bus_list_57, solution_generator_list_57, grid_ac_line_senders_list_57, grid_ac_line_receivers_list_57, grid_transformer_senders_list_57, grid_transformer_receivers_list_57, load_indices_list_57, shunt_indices_list_57, generator_indices_list_57, master_branch_list_57, pe_list_57 = process_grid_data(grid_bus_57, grid_generator_57, grid_load_57, grid_shunt_57, grid_ac_line_features_57, grid_ac_line_senders_57, grid_ac_line_receivers_57, grid_transformer_features_57, grid_transformer_senders_57, grid_transformer_receivers_57, solution_bus_57, solution_generator_57, solution_objective_57, grid_load_link_receivers_57, grid_load_link_senders_57, grid_shunt_link_receivers_57, grid_shunt_link_senders_57, grid_generator_link_senders_57, grid_generator_link_receivers_57, 57) 

In [ ]:
grid_bus_list_118, grid_generator_list_118, grid_load_list_118, grid_shunt_list_118, grid_ac_line_features_list_118, grid_transformer_features_list_118, solution_bus_list_118, solution_generator_list_118, grid_ac_line_senders_list_118, grid_ac_line_receivers_list_118, grid_transformer_senders_list_118, grid_transformer_receivers_list_118, load_indices_list_118, shunt_indices_list_118, generator_indices_list_118, master_branch_list_118, pe_list_118 = process_grid_data(grid_bus_118, grid_generator_118, grid_load_118, grid_shunt_118, grid_ac_line_features_118, grid_ac_line_senders_118, grid_ac_line_receivers_118, grid_transformer_features_118, grid_transformer_senders_118, grid_transformer_receivers_118, solution_bus_118, solution_generator_118, solution_objective_118, grid_load_link_receivers_118, grid_load_link_senders_118, grid_shunt_link_receivers_118, grid_shunt_link_senders_118, grid_generator_link_senders_118, grid_generator_link_receivers_118, 118) 

In [ ]:
grid_bus_list_500, grid_generator_list_500, grid_load_list_500, grid_shunt_list_500, grid_ac_line_features_list_500, grid_transformer_features_list_500, solution_bus_list_500, solution_generator_list_500, grid_ac_line_senders_list_500, grid_ac_line_receivers_list_500, grid_transformer_senders_list_500, grid_transformer_receivers_list_500, load_indices_list_500, shunt_indices_list_500, generator_indices_list_500, master_branch_list_500, pe_list_500 = process_grid_data(grid_bus_500, grid_generator_500, grid_load_500, grid_shunt_500, grid_ac_line_features_500, grid_ac_line_senders_500, grid_ac_line_receivers_500, grid_transformer_features_500, grid_transformer_senders_500, grid_transformer_receivers_500, solution_bus_500, solution_generator_500, solution_objective_500, grid_load_link_receivers_500, grid_load_link_senders_500, grid_shunt_link_receivers_500, grid_shunt_link_senders_500, grid_generator_link_senders_500, grid_generator_link_receivers_500, 500) 

In [ ]:
#### we have precomputed these encodings and will just load them here, computing them takes a long time for larger systems

In [ ]:
pe_list_14 = torch.load('bus_14_normed_pe_encodings.pt')

In [ ]:
pe_list_30 = torch.load('bus_30_normed_pe_encodings.pt')

In [ ]:
pe_list_57 = torch.load('bus_57_normed_pe_encodings.pt')

In [ ]:
pe_list_118 = torch.load('bus_118_normed_pe_encodings.pt')

In [ ]:
pe_list_500 = torch.load('bus_500_normed_pe_encodings.pt')

In [ ]:
del data_14
del data_30
del data_57
del data_118
del data_500
gc.collect()

In [ ]:
train_dataset_14 = make_training_datasets(grid_bus_list_14, grid_generator_list_14, grid_load_list_14, grid_shunt_list_14, grid_ac_line_features_list_14, grid_transformer_features_list_14, solution_bus_list_14, solution_generator_list_14, grid_ac_line_senders_list_14, grid_ac_line_receivers_list_14, grid_transformer_senders_list_14, grid_transformer_receivers_list_14, load_indices_list_14, shunt_indices_list_14, generator_indices_list_14, master_branch_list_14, pe_list_14)

In [ ]:
train_dataset_30 = make_training_datasets(grid_bus_list_30, grid_generator_list_30, grid_load_list_30, grid_shunt_list_30, grid_ac_line_features_list_30, grid_transformer_features_list_30, solution_bus_list_30, solution_generator_list_30, grid_ac_line_senders_list_30, grid_ac_line_receivers_list_30, grid_transformer_senders_list_30, grid_transformer_receivers_list_30, load_indices_list_30, shunt_indices_list_30, generator_indices_list_30, master_branch_list_30, pe_list_30)

In [ ]:
train_dataset_57 = make_training_datasets(grid_bus_list_57, grid_generator_list_57, grid_load_list_57, grid_shunt_list_57, grid_ac_line_features_list_57, grid_transformer_features_list_57, solution_bus_list_57, solution_generator_list_57, grid_ac_line_senders_list_57, grid_ac_line_receivers_list_57, grid_transformer_senders_list_57, grid_transformer_receivers_list_57, load_indices_list_57, shunt_indices_list_57, generator_indices_list_57, master_branch_list_57, pe_list_57)

In [ ]:
train_dataset_118 = make_training_datasets(grid_bus_list_118, grid_generator_list_118, grid_load_list_118, grid_shunt_list_118, grid_ac_line_features_list_118, grid_transformer_features_list_118, solution_bus_list_118, solution_generator_list_118, grid_ac_line_senders_list_118, grid_ac_line_receivers_list_118, grid_transformer_senders_list_118, grid_transformer_receivers_list_118, load_indices_list_118, shunt_indices_list_118, generator_indices_list_118, master_branch_list_118, pe_list_118)

In [ ]:
train_dataset_500 = make_training_datasets(grid_bus_list_500, grid_generator_list_500, grid_load_list_500, grid_shunt_list_500, grid_ac_line_features_list_500, grid_transformer_features_list_500, solution_bus_list_500, solution_generator_list_500, grid_ac_line_senders_list_500, grid_ac_line_receivers_list_500, grid_transformer_senders_list_500, grid_transformer_receivers_list_500, load_indices_list_500, shunt_indices_list_500, generator_indices_list_500, master_branch_list_500, pe_list_500)

In [ ]:
val_dataset_14 = make_validation_datasets(grid_bus_list_14, grid_generator_list_14, grid_load_list_14, grid_shunt_list_14, grid_ac_line_features_list_14, grid_transformer_features_list_14, solution_bus_list_14, solution_generator_list_14, grid_ac_line_senders_list_14, grid_ac_line_receivers_list_14, grid_transformer_senders_list_14, grid_transformer_receivers_list_14, load_indices_list_14, shunt_indices_list_14, generator_indices_list_14, master_branch_list_14, pe_list_14)

In [ ]:
val_dataset_30 = make_validation_datasets(grid_bus_list_30, grid_generator_list_30, grid_load_list_30, grid_shunt_list_30, grid_ac_line_features_list_30, grid_transformer_features_list_30, solution_bus_list_30, solution_generator_list_30, grid_ac_line_senders_list_30, grid_ac_line_receivers_list_30, grid_transformer_senders_list_30, grid_transformer_receivers_list_30, load_indices_list_30, shunt_indices_list_30, generator_indices_list_30, master_branch_list_30, pe_list_30)

In [ ]:
val_dataset_57 = make_validation_datasets(grid_bus_list_57, grid_generator_list_57, grid_load_list_57, grid_shunt_list_57, grid_ac_line_features_list_57, grid_transformer_features_list_57, solution_bus_list_57, solution_generator_list_57, grid_ac_line_senders_list_57, grid_ac_line_receivers_list_57, grid_transformer_senders_list_57, grid_transformer_receivers_list_57, load_indices_list_57, shunt_indices_list_57, generator_indices_list_57, master_branch_list_57, pe_list_57)

In [ ]:
val_dataset_118 = make_validation_datasets(grid_bus_list_118, grid_generator_list_118, grid_load_list_118, grid_shunt_list_118, grid_ac_line_features_list_118, grid_transformer_features_list_118, solution_bus_list_118, solution_generator_list_118, grid_ac_line_senders_list_118, grid_ac_line_receivers_list_118, grid_transformer_senders_list_118, grid_transformer_receivers_list_118, load_indices_list_118, shunt_indices_list_118, generator_indices_list_118, master_branch_list_118, pe_list_118)

In [ ]:
val_dataset_500 = make_validation_datasets(grid_bus_list_500, grid_generator_list_500, grid_load_list_500, grid_shunt_list_500, grid_ac_line_features_list_500, grid_transformer_features_list_500, solution_bus_list_500, solution_generator_list_500, grid_ac_line_senders_list_500, grid_ac_line_receivers_list_500, grid_transformer_senders_list_500, grid_transformer_receivers_list_500, load_indices_list_500, shunt_indices_list_500, generator_indices_list_500, master_branch_list_500, pe_list_500)

In [ ]:
train_dataset_combined = list(itertools.chain(train_dataset_14, train_dataset_30, train_dataset_57, train_dataset_118, train_dataset_500))
val_dataset_combined = list(itertools.chain(val_dataset_14, val_dataset_30, val_dataset_57, val_dataset_118, val_dataset_500))

In [ ]:
train_loader = DataLoader(train_dataset_combined, batch_size=batch_size, shuffle=True,num_workers=min(8, torch.get_num_threads()))
val_loader = DataLoader(val_dataset_combined, batch_size=batch_size, shuffle=False,num_workers=min(8, torch.get_num_threads()))

In [ ]:
del train_dataset_14
del train_dataset_30
del train_dataset_57
del train_dataset_118
del train_dataset_500
del val_dataset_14
del val_dataset_30
del val_dataset_57
del val_dataset_118
del val_dataset_500
gc.collect()

In [ ]:
class MLP(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size, layers, layernorm=True, use_leaky=False): 
        super().__init__()
        # Use Sequential instead of ModuleList for faster forward pass
        modules = []
        for i in range(layers):
            modules.append(torch.nn.Linear(
                input_size if i == 0 else hidden_size,
                output_size if i == layers - 1 else hidden_size,
            ))
            if i != layers - 1:
                modules.append(torch.nn.ReLU())
            if use_leaky:
                modules.append(torch.nn.LeakyReLU(negative_slope=0.02))
        if layernorm:
            modules.append(torch.nn.LayerNorm(output_size))
        
        self.network = torch.nn.Sequential(*modules)

    def forward(self, x):
        # Sequential is faster than iterating through ModuleList
        return self.network(x)

In [ ]:
# from performer_pytorch import SelfAttention
from torch_geometric.utils import to_dense_batch
from torch_geometric.nn.attention import PerformerAttention

class HeteroPerformerLayer(nn.Module):
    def __init__(self, hidden_dim, num_heads=1, dropout=0.0):
        super().__init__()
        self.hidden_dim = hidden_dim

        self.attn = PerformerAttention(
            channels=hidden_dim,
            heads=num_heads
        )
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.norm3 = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

        # Post-attention MLP
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.Linear(hidden_dim * 2, hidden_dim),
        )
    def forward(self, x_dict, xm_dict, batch_dict):
            # Flatten all node types
            flat_x, flat_xm, flat_batch = [], [], []
            slices = {}
            offset = 0
    
            for ntype in x_dict:
                x = x_dict[ntype]
                xm = xm_dict[ntype]
                b = batch_dict[ntype]  # batch indices for each node
    
                slices[ntype] = slice(offset, offset + x.size(0))
                flat_x.append(x)
                flat_xm.append(xm)
                flat_batch.append(b)
                offset += x.size(0)
    
            x_all = torch.cat(flat_x, dim=0)         # [N, D]
            xm_all = torch.cat(flat_xm, dim=0)       # [N, D]
            global_batch_all = torch.cat(flat_batch, dim=0) # [N]

            _, batch_all = torch.unique(global_batch_all, return_inverse=True)
            sorted_indices = torch.argsort(batch_all) # resort batch index to be ascending but that means I have to sort the xs myself too
            batch_all_sorted = batch_all[sorted_indices]
            x_all_sorted = x_all[sorted_indices]
            xm_all_sorted = xm_all[sorted_indices]
            # Convert to [B, N_max, D] and mask            
            x_dense, mask = to_dense_batch(x_all_sorted, batch_all_sorted)   # [B, N, D], [B, N]
            
            # Apply masked Performer attention
            x_attn = self.attn(x_dense, mask=mask)              # [B, N, D]
            # Residual + Norm
            xt_out = self.norm1(x_dense + x_attn)          
            # Unpad: [real_nodes, D]
            xt_out = xt_out[mask]
    
            x_comb = self.norm2(xt_out + xm_all_sorted)
            x_final = self.mlp(x_comb)

            # dont forget to give x_final it unsorted arrangement
            unsorted_x = torch.empty_like(x_final)
            unsorted_x[sorted_indices] = x_final
            x_final = unsorted_x
            # x_final = x_final[sorted_indices]

            # Unflatten by slice
            return {ntype: x_final[slices[ntype]] for ntype in x_dict.keys()}


In [ ]:
class HeteroInteractionNetwork(nn.Module):
    def __init__(self, node_types, edge_types,physical_edge_types, hidden_size, layers):
        super().__init__()
        
        self.physical_edge_types = physical_edge_types
        self.edge_updaters = nn.ModuleDict()
        for src, rel, dst in edge_types:
            edge_key = f"{src}_{rel}_{dst}"
            edge_type = (src, rel, dst)
            
            # For physical edges (ac_line and transformer), use node + edge features
            if edge_type in physical_edge_types:
                self.edge_updaters[edge_key] = MLP(hidden_size * 3, hidden_size, hidden_size, layers)
            else:
                # For other edge types, only use node features
                self.edge_updaters[edge_key] = MLP(hidden_size * 2, hidden_size, hidden_size, layers)
        
        # Create a node updater for each node type
        self.node_updaters = nn.ModuleDict({
            node_type: MLP(hidden_size * 2, hidden_size, hidden_size, layers)
            for node_type in node_types
        })

    
    def forward(self, x_dict, edge_indices_dict, edge_features_dict):
        # Store updated node and edge features
        updated_edge_features = {}
        
        # Prepare aggregated messages storage
        aggregated_messages = {node_type: torch.zeros_like(feat) 
                              for node_type, feat in x_dict.items()}
        
        # Process each edge type in parallel
        for edge_type, edge_index in edge_indices_dict.items():
            src_type, rel_type, dst_type = edge_type
            edge_key = f"{src_type}_{rel_type}_{dst_type}"
            
            # Get node features for this edge
            src, dst = edge_index
            x_i = x_dict[dst_type][dst]  # Destination nodes
            x_j = x_dict[src_type][src]  # Source nodes
            # Update edge features based on edge type
            if edge_type in self.physical_edge_types:
                # For physical edges, include edge features in message
                edge_feature = edge_features_dict[edge_type]
                edge_msg = torch.cat((x_i, x_j, edge_feature), dim=-1)
                updated_edge = self.edge_updaters[edge_key](edge_msg)
                updated_edge_features[edge_type] =  updated_edge + edge_feature  
            else:
                # For non-physical edges, only use node features
                edge_msg = torch.cat((x_i, x_j), dim=-1)
                updated_edge = self.edge_updaters[edge_key](edge_msg)
                
                #Check if we have existing edge features from previous layers
                # if edge_type in edge_features_dict:
                #     edge_feature = edge_features_dict[edge_type]
                #     updated_edge_features[edge_type] = edge_feature + updated_edge  ### removed residual connection for non-physical edges here
                # else:
                    # First layer - initialize with the computed edge features
                updated_edge_features[edge_type] = updated_edge 
            
            # Efficient message aggregation using torch_scatter
            aggregated_messages[dst_type] = torch_scatter.scatter_add(updated_edge, dst, dim=0, out=aggregated_messages[dst_type])
            aggregated_messages[src_type] = torch_scatter.scatter_add(updated_edge, src, dim=0, out=aggregated_messages[src_type])
        
        # Update node features
        updated_nodes = {}
        for node_type, x in x_dict.items():
            # Combine node features with aggregated messages
            node_input = torch.cat((x, aggregated_messages[node_type]), dim=-1)
            node_update = self.node_updaters[node_type](node_input)
            updated_nodes[node_type] = x + node_update 
        
        return updated_nodes, updated_edge_features


In [ ]:
class HeteroInteractGNN(torch.nn.Module):
    def __init__(
        self,
        hidden_size=256,
        n_mp_layers=5,
        bus_features=4,
        gen_features=11,
        load_features=2,
        shunt_features=2,
        ac_line_features=9,
        transformer_features=11,
        connects_to_features=3,
        output_dim=2
    ):
        super().__init__()
        
        # Define node and edge types
        self.node_types = ['bus', 'generator', 'load', 'shunt']
        self.edge_types = [
            ('bus', 'ac_line', 'bus'),
            ('bus', 'transformer', 'bus'),
            ('generator', 'connects_to', 'bus'),
            ('load', 'connects_to', 'bus'),
            ('shunt', 'connects_to', 'bus')
        ]

        self.physical_edge_types = [
            ('bus', 'ac_line', 'bus'),
            ('bus', 'transformer', 'bus')
        ]        
        #Node encoders - separate MLP for each node type
        self.node_encoders = nn.ModuleDict({
            'bus': MLP(bus_features, hidden_size, hidden_size-5, 2),
            'generator': MLP(gen_features, hidden_size, hidden_size-5, 2),
            'load': MLP(load_features, hidden_size, hidden_size-5, 2),
            'shunt': MLP(shunt_features, hidden_size, hidden_size-5, 2)
        })


        self.global_attn_layers = nn.ModuleList([
            HeteroPerformerLayer(hidden_size)
            for _ in range(n_mp_layers)
        ])

        #Edge encoders - separate MLP for each edge type
        self.edge_encoders = nn.ModuleDict({
            'ac_line': MLP(ac_line_features, hidden_size, hidden_size, 2),
            'transformer': MLP(transformer_features, hidden_size, hidden_size, 2)
        })
        
        # Interaction network layers
        self.n_mp_layers = n_mp_layers
        self.layers = torch.nn.ModuleList([
            HeteroInteractionNetwork(self.node_types, self.edge_types,self.physical_edge_types, hidden_size, 2)
            for _ in range(n_mp_layers)
        ])

        
        # Node decoders - separate for bus and generator
        self.node_decoders = nn.ModuleDict({
            'bus': MLP(hidden_size, hidden_size, output_dim, 2, layernorm=False),
            'generator': MLP(hidden_size, hidden_size, output_dim, 2,layernorm=False)
        })

    def forward(self, data):
        # Encode node features
        x_dict_init = {}
        x_dict = {}
        
        # Batch node encoding
        for node_type in self.node_types:
            if hasattr(data[node_type], 'x'):
                x_dict_init[node_type] = self.node_encoders[node_type](data[node_type].x)
                x_dict[node_type] = torch.cat([x_dict_init[node_type], data[node_type].pe], dim=-1)

        # Encode edge features
        edge_feature_dict = {}
        for src, rel, dst in self.edge_types:
            edge_type = (src, rel, dst)
            if (edge_type in self.physical_edge_types and 
                edge_type in data.edge_types and 
                hasattr(data[edge_type], 'edge_attr')):
                edge_feature_dict[edge_type] = self.edge_encoders[rel](data[edge_type].edge_attr)
        
        # Extract edge indices
        edge_index_dict = {
            edge_type: data[edge_type].edge_index
            for edge_type in self.edge_types
            if edge_type in data.edge_types and hasattr(data[edge_type], 'edge_index')
        }

        batch_dict = {
            ntype: data[ntype].graph_id
            for ntype in x_dict
        }
        

        # Apply message passing layers
        for i in range(self.n_mp_layers):
            xm_dict, edge_feature_dict = self.layers[i](x_dict, edge_index_dict, edge_feature_dict)
            x_dict = self.global_attn_layers[i](x_dict, xm_dict, batch_dict)

        # Apply decoders
        output = {}
        output['bus'] = torch.sigmoid(self.node_decoders['bus'](x_dict['bus']))
        output['generator'] = torch.sigmoid(self.node_decoders['generator'](x_dict['generator']))
        
        return output

In [ ]:
## wandb set-up
api_key = '' ###insert your own api keys
wandb.login(key=api_key)

In [ ]:
def convert_voltage_bounds(model_input):

    num_nodes = model_input.shape[0]

    vmin = model_input[:,2:3]

    vmax = model_input[:,3:4]

    thetamin =  torch.tensor([-2.00]).to(device)
    thetamin = thetamin.tile((num_nodes,1))

    thetamax =  torch.tensor([2.00]).to(device)
    thetamax = thetamax.tile((num_nodes,1))

    bounds_up = torch.concat((thetamax, vmax), dim=1)
    bounds_down = torch.concat((thetamin, vmin),dim=1)


    return bounds_up, bounds_down
    

In [ ]:
def convert_power_bounds(model_input):

    num_nodes = model_input.shape[0]

    pmin = model_input[:,2:3]

    pmax = model_input[:,3:4]

    
    qmin = model_input[:,5:6]

    qmax = model_input[:,6:7]

    bounds_up = torch.concat((pmax, qmax), dim=1)
    bounds_down = torch.concat((pmin, qmin),dim=1)


    return bounds_up, bounds_down
    

In [ ]:
# Example training step
def train_model(model, trainloader, optimizer):
    
    model.train()
    total_loss = 0
    criterion = nn.MSELoss()
    
    for batch in trainloader:
    
        optimizer.zero_grad()
        
        batch = batch.to(device)

        # Forward pass
        pred_dict = model(batch)

        voltage_up, voltage_down = convert_voltage_bounds(batch['bus'].x)
        voltage_up = voltage_up.to(device)
        voltage_down = voltage_down.to(device)
        voltages = pred_dict['bus'] * (voltage_up - voltage_down) + voltage_down
        voltages = torch.clamp(voltages,min=voltage_down, max=voltage_up)

        power_up, power_down = convert_power_bounds(batch['generator'].x)
        power_up = power_up.to(device)
        power_down = power_down.to(device)
        powers = pred_dict['generator'] * (power_up - power_down) + power_down
        powers = torch.clamp(powers,min=power_down, max=power_up)
        

        combined_targets = torch.cat([batch['bus'].y, batch['generator'].y], dim=0)
        combined_outputs = torch.cat([voltages, powers], dim=0)
        loss = criterion(combined_targets, combined_outputs)

        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    return total_loss / len(trainloader)

In [ ]:
# Example training step
def validate_model(model, val_loader):
    
    model.eval()
    total_loss = 0
    criterion = nn.MSELoss()
    
    for batch in val_loader:
        
        batch = batch.to(device)

        # Forward pass
        pred_dict = model(batch)

        voltage_up, voltage_down = convert_voltage_bounds(batch['bus'].x)
        voltage_up = voltage_up.to(device)
        voltage_down = voltage_down.to(device)
        voltages = pred_dict['bus'] * (voltage_up - voltage_down) + voltage_down
        voltages = torch.clamp(voltages,min=voltage_down, max=voltage_up)

        power_up, power_down = convert_power_bounds(batch['generator'].x)
        power_up = power_up.to(device)
        power_down = power_down.to(device)
        powers = pred_dict['generator'] * (power_up - power_down) + power_down
        powers = torch.clamp(powers,min=power_down, max=power_up)

        combined_targets = torch.cat([batch['bus'].y, batch['generator'].y], dim=0)
        combined_outputs = torch.cat([voltages, powers], dim=0)

        loss = criterion(combined_targets, combined_outputs)
     
        total_loss += loss.item()
        
    return total_loss / len(val_loader)

In [ ]:
@torch.no_grad()
def test_model(model, testloader):

    model.eval()
    criterion = nn.MSELoss()
    
    total_loss = 0.0
    voltage_predictions = []
    voltage_targets = []
    power_predictions = []
    power_targets = []
    
    for batch in testloader:

        batch = batch.to(device)

        # Forward pass
        pred_dict = model(batch)

        voltage_up, voltage_down = convert_voltage_bounds(batch['bus'].x)
        voltage_up = voltage_up.to(device)
        voltage_down = voltage_down.to(device)
        voltages = pred_dict['bus'] * (voltage_up - voltage_down) + voltage_down
        voltages = torch.clamp(voltages,min=voltage_down, max=voltage_up)

        power_up, power_down = convert_power_bounds(batch['generator'].x)
        power_up = power_up.to(device)
        power_down = power_down.to(device)
        powers = pred_dict['generator'] * (power_up - power_down) + power_down
        powers = torch.clamp(powers,min=power_down, max=power_up)
        
        combined_targets = torch.cat([batch['bus'].y, batch['generator'].y], dim=0)
        combined_outputs = torch.cat([voltages, powers], dim=0)
        loss = criterion(combined_targets, combined_outputs)
        
        total_loss += loss.item()

        # Store predictions and targets for overall metrics
        voltage_predictions.append(voltages.cpu())
        voltage_targets.append(batch['bus'].y.cpu())

        power_predictions.append(powers.cpu())
        power_targets.append(batch['generator'].y.cpu())

    
    return total_loss / len(testloader), voltage_predictions, voltage_targets, power_predictions, power_targets

In [ ]:
run = wandb.init(
      # Set the project where this run will be logged
      project="Towards_Generalization_of_GNN_for_ACOPF",
      # We pass a run name (otherwise it’ll be randomly assigned, like sunshine-lollypop-10)
      name=f"Pretraining_5grids_with_HybridHeteroGNN_5_256_PQVT",
      # Track hyperparameters and run metadata
      config={
      "architecture": "GNN",
      "dataset": "N-1",
      "epochs": 100,
      })

In [ ]:
model = HeteroInteractGNN().to(device)

In [ ]:
tot_params = 0
for parameter in model.parameters():
  layer_ws = 1
  for val in parameter.shape:
      layer_ws*=val
  tot_params += layer_ws
print(f"Total number of parameters = {tot_params}")

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5,weight_decay=5e-8)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=20)

In [ ]:
from pathlib import Path
def save_checkpoint(model, optimizer, scheduler, epoch, loss):
    """
    Save model checkpoint including all training state
    """
    # Create checkpoint directory if it doesn't exist
    # Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)
    
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
        'loss': loss,
        'learning_rate': optimizer.param_groups[0]['lr']  # Current LR
    }
    
    # Save with epoch number in filename
    checkpoint_path = f'pretraining_checkpoint_epoch_{epoch}.pth'
    torch.save(checkpoint, checkpoint_path)
    
    # Also save as 'latest' for easy loading
    latest_path = 'pretraining_checkpoint_latest.pth'
    torch.save(checkpoint, latest_path)
    
    print(f"Checkpoint saved at epoch {epoch}: {checkpoint_path}")

def load_checkpoint(model, optimizer, scheduler, checkpoint_path):
    """
    Load checkpoint and restore training state
    """
    if not os.path.exists(checkpoint_path):
        print(f"No checkpoint found at {checkpoint_path}")
        return 0
    
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # Load model state
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Load optimizer state
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    # Load scheduler state if it exists
    if scheduler and checkpoint['scheduler_state_dict']:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    
    epoch = checkpoint['epoch']
    loss = checkpoint['loss']
    lr = checkpoint['learning_rate']
    
    print(f"Loaded checkpoint from epoch {epoch}, loss: {loss:.4f}, lr: {lr:.6f}")
    
    return epoch + 1  # Return next epoch to start from

In [ ]:
training_losses = []
validation_losses = []
best_valid_loss = float('inf')
early_stop_thresh = 100
best_epoch = -1
best_model_state = None
num_epochs = 100

for epoch in tqdm(range(num_epochs), desc="Training Progress"):
    train_loss = train_model(model, train_loader, optimizer)
    valid_loss = validate_model(model, val_loader)
    training_losses.append(train_loss)
    validation_losses.append(valid_loss)

    wandb.log({"training_loss": train_loss, "validation_loss": valid_loss})

    scheduler.step(train_loss)

    if epoch % 10 == 0:
      print(f'Epoch: {epoch}')
      print(f'\tTrain Loss: {train_loss:.4f}')
      print(f'\t Val. Loss: {valid_loss:.4f}')
    if valid_loss < best_valid_loss:
      best_valid_loss = valid_loss
      best_model_state = deepcopy(model.state_dict())

    if epoch % 10 == 0:
        save_checkpoint(model, optimizer, scheduler, epoch, valid_loss)


plt.subplots(figsize=(5,3))
plt.plot([i for i in range(len(training_losses))], training_losses, 'r', label='Training loss')
plt.plot([i for i in range(len(validation_losses))], validation_losses, 'g', label='Validation loss')
plt.legend()
plt.title(f'GNN Training and Validation loss',fontsize = 15)
plt.xlabel('Epochs',fontsize = 12)
plt.ylabel('MSE Loss',fontsize = 12)
plt.semilogy()

training_losses=np.array(training_losses)
validation_losses=np.array(validation_losses)

model.load_state_dict(best_model_state)
model.eval()

In [ ]:
torch.save(model.state_dict(), f"Five_systems_Pretraining_HybridHeteroGNN_5_256_PQVT.pth")
wandb.save(f"Five_systems_bus_Pretraining_HybridHeteroGNN_5_256_PQVT.pth")  # Upload to WandB

In [ ]:
wandb.finish()

In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()

In [ ]:
print('hurray, done!')